Create video manifest, hindi makapal

In [1]:
from pathlib import Path
import pandas as pd
import os

ROOT = Path.cwd().parents[1]

LABELS_PATH = ROOT / "csv\expanded_labels.csv"

labels = pd.read_csv(LABELS_PATH)

print(labels.shape)
labels.head()

(105, 6)


,id,label,display_label,category,modality,enabled
0,0,GOOD MORNING,Good Morning,GREETING,dynamic,True
1,1,GOOD AFTERNOON,Good Afternoon,GREETING,dynamic,True
2,2,GOOD EVENING,Good Evening,GREETING,dynamic,True
3,3,HELLO,Hello,GREETING,dynamic,True
4,4,HOW ARE YOU,How Are You,GREETING,dynamic,True


In [2]:
from pathlib import Path

RAW = ROOT / "dynamic_raw"
print(RAW)
print(RAW.exists())

manifest= []

for _, row in labels.iterrows():

    label_id = row["id"]

    folder = RAW/str(label_id)

    if not folder.exists():
        continue

    videos = sorted([
        f for f in folder.iterdir()
        if f.suffix.lower()
        in [".mp4", ".avi", ".mov", ".mkv"]
    ])

    for take, video in enumerate(videos, start=1):

        manifest.append({

            "video_id":
                f"{label_id:03d}_{take:03d}",

            "video_path":
                str(video),

            "label_id":
                label_id,

            "label":
                row["label"],

            "display_label":
                row["display_label"],

            "category":
                row["category"],

            "modality":
                row["modality"],

            "signer_id":
                "unknown",

            "take":
                take,

            "enabled":
                row["enabled"]
        })

c:\Projects\signia-fsl-recognition\dynamic_raw
True


In [3]:
manifest = pd.DataFrame(manifest)

manifest.head()

,video_id,video_path,label_id,label,display_label,category,modality,signer_id,take,enabled
0,000_001,c:\Projects\signia-fsl-recognition\dynamic_raw...,0,GOOD MORNING,Good Morning,GREETING,dynamic,unknown,1,True
1,000_002,c:\Projects\signia-fsl-recognition\dynamic_raw...,0,GOOD MORNING,Good Morning,GREETING,dynamic,unknown,2,True
2,000_003,c:\Projects\signia-fsl-recognition\dynamic_raw...,0,GOOD MORNING,Good Morning,GREETING,dynamic,unknown,3,True
3,000_004,c:\Projects\signia-fsl-recognition\dynamic_raw...,0,GOOD MORNING,Good Morning,GREETING,dynamic,unknown,4,True
4,000_005,c:\Projects\signia-fsl-recognition\dynamic_raw...,0,GOOD MORNING,Good Morning,GREETING,dynamic,unknown,5,True


In [6]:
import cv2

fps_list = []
frame_list = []
duration_list = []


for path in manifest["video_path"]:

    cap = cv2.VideoCapture(path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    frames = cap.get(cv2.CAP_PROP_FRAME_COUNT)

    duration = 0
    if fps > 0:
        duration = frames / fps

    fps_list.append(fps)
    frame_list.append(int(frames))
    duration_list.append(duration)

    cap.release()

manifest["fps"] = fps_list
manifest["frames"] = frame_list
manifest["duration"] = duration_list

In [8]:
OUTPUT = ROOT / "video_manifest.csv"

manifest.to_csv(
    OUTPUT,
    index=False
)

print(
    "saved:",
    OUTPUT
)

saved: C:\Projects\signia-fsl-recognition\video_manifest.csv


In [9]:
manifest.groupby(
    "category"
).size()

category
CALENDAR         247
COLOR            261
DAYS             201
DRINK            203
FAMILY           200
FOOD             200
GREETING         206
NUMBER           202
RELATIONSHIPS    202
SURVIVAL         208
dtype: int64